# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [1]:
%load_ext autoreload
%autoreload 2

from campaign_lib import *

# --- Services ---
svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
# ALL experiment knobs live here — no hidden defaults in service code.
campaign_config = {
    "sample_size": 15,              # queries per eval step (0 = all)
    "exploration_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "exploration_rate": 0.5,        # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "exclude_nodes": ["llm_ranking"],    # nodes to skip (e.g. ["entity_profiling"])
    # --- Backend node overrides ---
    # Override values from GET /pipeline. Flat names resolved via schema override_map.
    # See show_pipeline_snapshot() output for available params per node.
    "pipeline_overrides": {
        # web_search
        "max_sites": 20,                          # max web pages to fetch content from
        "num_results": 20,                        # search engine results to request
        "content_char_limit": 800,                # max chars per web page
        # entity_profiling
        "profiling_model": "openai/gpt-oss-120b", # entity_profiling LLM model
        "profiling_max_tokens": 4000,             # entity_profiling max output tokens
        # llm_ranking
        "ranking_model": "openai/gpt-oss-120b",   # llm_ranking LLM model
    },
    "optimization": {
        # --- Core loop ---
        "patience": 2,                   # consecutive non-improvements before stop/escalate
        "max_rounds": None,              # None = unlimited
        "n_variants": 5,                 # candidates per round
        "creativity": 0.7,               # temperature for candidate generation
        "improvement_threshold": 0.01,   # accuracy delta to count as improvement
        "seed": 42,                      # subsampling seed (reproducibility)
        "max_failures": 15,              # failure examples fed to LLM candidate generation
        # --- Escalation ---
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": 2,               # L2 stalls before L3
        "l3_patience": 1,               # L3 stalls before stop
        "l2_temperature": 0.3,           # LLM temperature for L2 transitions
        "l3_temperature": 0.5,           # LLM temperature for L3 transitions
        # --- Critique ---
        "enable_critique": True,         # critique agent between generate/evaluate
    },
    "eval_llm": {
        # "model":       "moonshotai/kimi-k2-instruct-0905",  # 10x more expensive
        "model":       "openai/gpt-oss-120b",
        "provider":    "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None        # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

Backend: http://127.0.0.1:8000


2026-04-01 14:25:55 INFO     [api.services.pipeline_discovery] Parsed pipeline 'TermNorm' with 6 steps
2026-04-01 14:25:55 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Pipeline: termnorm (6 nodes)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md
PIPELINE SNAPSHOT: TermNorm v1.1
--------------------------------------------------
  Nodes:   ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']
{
  "name": "TermNorm",
  "version": "v1.1",
  "description": "TermNorm AI terminology normalization pipeline",
  "required_step": "entity_profile",
  "template_variables": [
    "{{core_concept}}",
    "{{entity_profile_json}}",
    "{{matches}}"
  ],
  "dataset_name": "termnorm_ground_truth",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-

In [2]:
#@title Load data & evaluation context
RUN_BASELINE = False

train_data, session_terms = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
svc["session_terms"] = session_terms

baseline_ps, eval_data, campaign_rounds, baseline_results = await prepare_eval_context(
    svc, train_data, campaign_config, run_baseline=RUN_BASELINE,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets

Evaluation data: 984 queries


In [3]:
#@title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')

pipeline_params = show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    baseline_prompt_fields=campaign_rounds[0]["prompt_fields"].model_dump() if campaign_rounds else None,
)


  EXPERIMENT DASHBOARD (termnorm-local)
  Dataset runs: 208 total (2 77e7e77777e7, 2 be8c4d8229f6, 1 dbb05e948b53, 66 e9315d106538, 137 other)
  Best result: 66.7% (scan)

  Campaigns (most recent first):
    cycle_e9315d106538  completed    7 rounds  best=0.0%  base=0.0%
      patience=?  rounds=None  sample=15  model=openai/gpt-oss-120b

    cycle_be8c4d8229f6  interrupted  0 rounds  best=33.3%  base=33.3%
      patience=2  rounds=None  sample=15  model=openai/gpt-oss-120b

    cycle_dbb05e948b53  interrupted  0 rounds  best=33.3%  base=33.3%
      patience=2  rounds=None  sample=15  model=moonshotai/kimi-k2-instruct-09

  Set experiment_id="<short_id>" to see full config and diff
  Active: cycle_e2c05b4bcc16



## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [4]:
#@title Task context + scan advisor
ADVISOR_MODEL = "openai/gpt-oss-120b"  # model for scan advisor LLM call

task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

# preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)
# advisory, scan_variants, schema_labels = await run_scan_advisor(
#     campaign_config, svc,
#     task_description=task_context,
#     model=ADVISOR_MODEL,
# )

TASK CONTEXT DECOMPOSITION
--------------------------------------------------
  domain: Life Cycle Assessment (LCA) terminology and database normalization
  pipeline_purpose: Normalize free-form user inputs into exact, standardized entries of LCA databases (e.g., ecoinvent, GaBi) for accurate impact assessment.
  data_characteristics: Textual inputs of varying length containing industrial material codes, DIN/ISO/IEC references, brand/trade names, chemical formulas, agricultural qualifiers, geographic tags, and mixed-language terms; multilingual (German, French, English); high variability, typically dozens to hundreds of entries per project.
  optimization_goals: Maximize correct match rate (precision and recall) against ground truth, correctly flag no‑match cases, handle synonyms, variants, and geographic scopes, and minimize false positives.
  key_challenges: Structural mismatch between unstructured shorthand and rigid database naming; need for domain knowledge to decode brands, stand

In [5]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 3  # queries per scan variant (0 = use all)

scan_variants = {
    # ── Token matching ───────────────────────────────────────────────────
    'max_token_candidates': [10, 30, 50],
    # ── Web search: query framing ────────────────────────────────────────
    'query_prefix': [
        # --- Database-oriented ---
        'ecoinvent', 'GaBi', 'ecoinvent LCA database material name',
        'material composition LCA', # 'ecoinvent equivalent', 'ecoinvent match for', 'identify LCA material for',
        'what is',# 'define', 'identify', 'describe material',
        # 'chemical composition of', 'CAS number', 'IUPAC name for',
        'technical data sheet', 'product specification', # 'manufacturer datasheet', 'material safety data sheet',
        'manufactured from', 'production process for', # 'raw material for',
        # --- Synonym / translation ---
        # 'also known as', # 'synonym for', 'equivalent material', 'alternative name for',
        # 'wikipedia', # 'material properties of',
    ],
    # ── Web search: volume knobs ─────────────────────────────────────────
    'max_sites': [3, 7, 12],
    'num_results': [5, 20, 40],
    'content_char_limit': [400, 800, 1500],
    # ── Fuzzy matching ───────────────────────────────────────────────────
    # 'fuzzy_threshold': [50, 70, 90],
    # 'fuzzy_scorer': ['ratio', 'WRatio', 'token_set_ratio'],
    # ── Entity profiling: LLM tuning ─────────────────────────────────────
    'profiling_temperature': [0.0, 0.3, 0.7],
    'raw_content_limit': [1000, 2500, 8000],
    # ── Prompt fields ────────────────────────────────────────────────────
    'thinking_style': [
          'Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.',
          'Consider the most likely interpretation first, then check alternatives.',       
          'Reason by elimination: discard obviously wrong candidates, then rank the rest.',
      ],
    # ── Entity profiling: schema mutations ───────────────────────────────
    'profiling_schema': [
        # --- LCA-database-specific (original) ---
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'],
         ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]],
        [['-', 'manufacturing_processes'], ['-', 'applications'],
         ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        # [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]],
        [['-', 'applications'], ['+', 'ecoinvent_candidate_names', 'array', True, 'Exact ecoinvent activity names this entity most likely maps to']],
        [['-', 'manufacturing_processes'], ['+', 'database_search_tokens', 'array', True, 'Optimized search tokens for LCA database lookup including spelling variants']],
        # [['~', 'classification_aliases', 'classification_aliases', 'array', True, 'All valid ecoinvent/GaBi naming variants including geography codes and system model suffixes']],
        # --- Minimalist: strip to core matching signals ---
        [['-', 'applications'], ['-', 'manufacturing_processes'], ['-', 'notes'], ['-', 'technical_specifications']],
        # --- Chemical identity ---
        [['+', 'cas_number', 'string', False, 'CAS registry number if identifiable from context'],
         ['+', 'chemical_formula', 'string', False, 'Chemical formula or molecular structure notation']],
        # --- Trade name decoding ---
        [['~', 'key_properties', 'trade_names', 'array', True, 'Known commercial/trade names and brand names for this material, e.g. Makrolon=polycarbonate, Delrin=POM']],
        # --- Material hierarchy (specific→generic) ---
        [['+', 'material_hierarchy', 'array', False, 'Classification chain from specific to generic, e.g. [Makrolon 2805, polycarbonate, thermoplastic, polymer]']],
        # --- Process-centric (flip perspective from material to process) ---
        [['~', 'applications', 'production_route', 'string', False, 'Primary production/manufacturing route e.g. injection molding, extrusion, casting'],
         ['~', 'notes', 'form_factor', 'string', False, 'Physical form: granulate, sheet, rod, wire, powder, liquid, film']],
        # --- Standards-focused ---
        [['+', 'applicable_standards', 'array', False, 'DIN/ISO/EN/ASTM standards that reference or define this material'],
         ['-', 'applications']],
        # # --- Geography-aware ---
        # [['+', 'supply_chain_geography', 'string', False, 'Most likely geographic origin or market region for this material']],
        # # --- Confidence / ambiguity signal ---
        # [['+', 'confidence_level', 'string', False, 'How confident the model is in the identification: high/medium/low/ambiguous'],
        #  ['+', 'ambiguity_notes', 'string', False, 'What makes this input hard to identify — abbreviation, trade name, multi-material, etc.']],
        # # --- Spelling / language variant boost ---
        # [['+', 'spelling_variants', 'array', False, 'All known spelling variants across EN/DE/FR, e.g. aluminium/aluminum, polyamid/polyamide'],
        #  ['-', 'notes']],
        # --- Werkstoff / alloy code decoding ---
        [['+', 'material_code_decoded', 'string', False, 'Decoded meaning of any material code, Werkstoff number, or alloy designation present in the input'],
         ['+', 'base_material', 'string', False, 'The fundamental base material, e.g. brass, steel, polycarbonate']],
        # --- Functional equivalence ---
        [['~', 'applications', 'functional_unit', 'string', False, 'The functional unit this material serves, e.g. structural plastic, electrical insulation, food-grade packaging'],
         ['+', 'substitutes', 'array', False, 'Materials that could serve the same functional role']],
    ],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['ecoinvent', 'GaBi', 'ecoinvent LCA database material name', 'material composition LCA', 'what is', 'technical data sheet', 'product specification', 'manufactured from', 'production process for']
  max_sites: [3, 7, 12]
  num_results: [5, 20, 40]
  content_char_limit: [400, 800, 1500]
  profiling_temperature: [0.0, 0.3, 0.7]
  raw_content_limit: [1000, 2500, 8000]
  thinking_style: ['Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.', 'Consider the most likely interpretation first, then check alternatives.', 'Reason by elimination: discard obviously wrong candidates, then rank the rest.']
  profiling_schema: (baseline + 13 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this e

In [6]:
#@title Run sensitivity scan
scan_baseline_sp, scan_df, axis_profiles = await run_sensitivity_scan(
    baseline_ps, campaign_config, scan_variants, eval_data,
    scan_sample_size=scan_sample_size,
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Active nodes: cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching  Excluded: llm_ranking
Restructured baseline fields (cached):
  persona: You are a comprehensive technical database API specialized in exhaustive entity ...
  task_intent: Extract all possible information about a given query from research data and retu...
  problem_description: Given a product or technical description query, parse unstructured research text...
  instruction: Extract ALL possible information about '{{query}}' from the research data and re...
  thinking_style: Think step by step, first gather explicit data, then infer implicit attributes, ...
  answer_format: A single JSON object matching the provided {{format_string}}, with all array fie...
Search baseline: 822ab596df90 (render: 1541 chars)
Historical data: 171 results across 17 unique prompts
Matching runs (step sequence): 128, 53 cached results
Scan variant coverage (128 matching runs):
max_token_candidates     10→28 ✓  30→24 ✓  5

2026-04-01 14:25:59 INFO     [api.services.search.sensitivity_scanner] Pruning axis 'query_prefix': all 2 variants overlap with baseline CI — skipping 7 remaining values


  [1] GaBi                                       0/3  0.0%  +0.0% [cached]
  >> PRUNED: query_prefix — 2 variants overlap baseline CI, skipping 7 remaining

  Axis 2/9: max_sites (pipeline_param, 3 values)
 16.1s MISS 3/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Polyamide (Nylon) 6.6/EU-27
  8.9s MISS 3/20  [token] 📖  Stainless steel EN 10270-3/winding             -> Steel, chromium steel 18/8 {GLO}| m
 12.0s MISS 2/20  [token] 📖  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop
  [0] 3                                          0/3  0.0%  +0.0% [cached]
 10.3s MISS 3/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Polyamide (Nylon) 6.6/EU-27
 11.5s MISS 2/20  [token] 📖  Stainless steel EN 10270-3/winding             -> Hot rolling, steel {RoW}| hot rolli
 15.4s MISS 2/20  [token] 📖  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop
            ⚠ web_search: 6 of 20 fetched URL

2026-04-01 14:25:59 INFO     [api.services.search.sensitivity_scanner] Pruning axis 'max_sites': all 2 variants overlap with baseline CI — skipping 1 remaining values


  [1] 7                                          0/3  0.0%  +0.0% [cached]
  >> PRUNED: max_sites — 2 variants overlap baseline CI, skipping 1 remaining

  Axis 3/9: num_results (pipeline_param, 3 values)
 11.7s MISS 3/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Polyamide (Nylon) 6.6/EU-27
            ⚠ web_search: 1 of 5 fetched URLs returned content (4 filtered: 2×http_429, 2×skip_extension)
  9.2s MISS 2/20  [token] 📖  Stainless steel EN 10270-3/winding             -> Steel, chromium steel 18/8 {GLO}| m
            ⚠ web_search: 4 of 5 fetched URLs returned content (1 filtered: 1×skip_extension)
 10.4s MISS 3/20  [token] 📖  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop
            ⚠ web_search: 1 of 5 fetched URLs returned content (4 filtered: 3×too_short, 1×http_403)
  [0] 5                                          0/3  0.0%  +0.0% [cached]
 10.3s MISS 3/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Pol

...

  [1] Consider the most likely interpretation ... 1/3  33.3%  +30.0% ^ [cached]
  >> PRUNED: thinking_style — 2 variants overlap baseline CI, skipping 1 remaining

  Axis 8/9: profiling_schema (pipeline_param, 14 values)
  8.6s MISS 2/20 web📖[][token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Polyamide (Nylon) 6.6/EU-27
  3.9s MISS 2/20 web📖[][token] 📖  Stainless steel EN 10270-3/winding             -> Steel, chromium steel 18/8 {GLO}| m
  5.9s MISS 6/20 web📖[][token] 📖  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop
            ⚠ web_search: 6 of 6 fetched URLs returned content
  [0] schema(11 fields)                          0/3  0.0%  +0.0% [cached]
  7.9s MISS 5/20 web📖[][token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Polyamide (Nylon) 6.6/EU-27
  5.8s MISS 2/20 web📖[][token] 📖  Stainless steel EN 10270-3/winding             -> Steel, chromium steel 18/8 {GLO}| m
  9.5s MISS 2/20 web📖[][token] 📖  SJRG0010-ABS/molding   

2026-04-01 14:25:59 INFO     [api.services.search.sensitivity_scanner] Pruning axis 'profiling_schema': all 2 variants overlap with baseline CI — skipping 12 remaining values


  [1] schema(13 fields)                          0/3  0.0%  +0.0% [cached]
  >> PRUNED: profiling_schema — 2 variants overlap baseline CI, skipping 12 remaining

  Axis 9/9: max_token_candidates (pipeline_param, 3 values)
  5.2s HIT  web📖[][token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75
  4.2s HIT  web📖[][token] 📖  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra
  4.0s MISS --/10 web📖[][token] 📖  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop
            ⚠ web_search: 6 of 6 fetched URLs returned content
  [0] 10                                         2/3  66.7%  +58.3% ^ [cached]
  4.7s MISS 2/30 web📖[][token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 50
  8.8s MISS 3/30 web📖[][token] 📖  Stainless steel EN 10270-3/winding             -> Steel, chromium steel 18/8 {GLO}| m
  4.7s MISS 4/30 web📖[][token] 📖  SJRG0010-AB

2026-04-01 14:25:59 INFO     [api.services.search.sensitivity_scanner] Pruning axis 'max_token_candidates': all 2 variants overlap with baseline CI — skipping 1 remaining values


  [1] 30                                         0/3  0.0%  +0.0% [cached]
  >> PRUNED: max_token_candidates — 2 variants overlap baseline CI, skipping 1 remaining
  >> max_token_candidates: range=58.3%, best=+58.3%, worst=+0.0%, budget=high
  >> thinking_style: range=31.7%, best=+30.0%, worst=-1.7%, budget=skip
  >> query_prefix: range=30.0%, best=+30.0%, worst=+0.0%, budget=skip
  >> profiling_temperature: range=30.0%, best=+30.0%, worst=+0.0%, budget=skip
  >> raw_content_limit: range=30.0%, best=+30.0%, worst=+0.0%, budget=skip
  >> max_sites: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip
  >> num_results: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip
  >> content_char_limit: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip
  >> profiling_schema: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip

Sensitivity scan complete: 18 variants evaluated

Rank  Axis                      Type            Card  Range    Budget  
-------------------------------------------------------------

In [7]:
#@title Scan analytics
difficulty_df = show_scan_analytics(scan_df, axis_profiles, svc)

VARIANT LEADERBOARD (all scan combos)


,rank,axis,variant,accuracy,delta,hits/total,errors
0,1,max_token_candidates,10,66.7%,+58.3%,2/3,0
1,2,query_prefix,ecoinvent,33.3%,+30.0%,1/3,0
2,3,raw_content_limit,2500,33.3%,+30.0%,1/3,0
3,4,profiling_temperature,0.0,33.3%,+30.0%,1/3,0
4,5,thinking_style,Consider the most likely interpretation ...,33.3%,+30.0%,1/3,0
5,6,query_prefix,GaBi (baseline),0.0%,-,0/3,0
6,7,num_results,20 (baseline),0.0%,-,0/3,0
7,8,max_sites,3 (baseline),0.0%,-,0/3,0
8,9,num_results,5 (baseline),0.0%,-,0/3,0
9,10,max_sites,7 (baseline),0.0%,-,0/3,0



PER-AXIS STATISTICS


,axis,type,variants,mean_acc,std_acc,best_acc,worst_acc,sensitivity,budget
0,query_prefix,pipeline_param,2,16.7%,23.6%,33.3%,0.0%,0.300,skip
1,max_sites,pipeline_param,2,0.0%,0.0%,0.0%,0.0%,0.000,skip
2,num_results,pipeline_param,2,0.0%,0.0%,0.0%,0.0%,0.000,skip
3,content_char_limit,pipeline_param,2,0.0%,0.0%,0.0%,0.0%,0.000,skip
4,profiling_temperature,pipeline_param,2,16.7%,23.6%,33.3%,0.0%,0.300,skip
5,raw_content_limit,pipeline_param,2,16.7%,23.6%,33.3%,0.0%,0.300,skip
6,thinking_style,prompt_field,2,16.7%,23.6%,33.3%,0.0%,0.317,skip
7,profiling_schema,pipeline_param,2,0.0%,0.0%,0.0%,0.0%,0.000,skip
8,max_token_candidates,pipeline_param,2,33.3%,47.1%,66.7%,0.0%,0.583,high


QUERY DIFFICULTY (10 queries across 139 scan runs)
  easy: 0 (0%) | discriminating: 5 (50%) | hard: 5 (50%) | error: 0 (0%)



,query,ground_truth,hit_rate,hits/evals,error_rate,classification
0,Copper Wire/cold forming,"Metal working, average for copper product manu...",0.000000,0/4,0.000000,hard
1,PC GF10 makrolon material/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/4,0.000000,hard
2,PA6/66 Ultramid C3U/molding,Injection moulding {RER}| injection moulding |...,0.000000,0/5,0.000000,hard
3,PA 66 25% GF V0 RAL 7012/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/5,0.000000,hard
4,EN 10270-3-1.4568\nX7CrNi17-7 (DIN17224 4.4568...,"Sheet rolling, chromium steel {RER}| sheet rol...",0.000000,0/4,0.000000,hard
5,SJRG0010-ABS/molding,Injection moulding {RER}| injection moulding |...,0.038760,5/129,0.023256,discriminating
6,Kingfa NPG25,Glass fibre reinforced plastic | 75% PA66 25% ...,0.200000,1/5,0.000000,discriminating
7,SJRG0013-PA/molding,Injection moulding {RER}| injection moulding |...,0.333333,1/3,0.000000,discriminating
8,PA66-GF25 ULTRAMID A3UG5 RAL7035 grey,Glass fibre reinforced plastic | 75% PA66 25% ...,0.364964,50/137,0.029197,discriminating
9,Stainless steel EN 10270-3/winding,"Wire drawing, steel {RER}| wire drawing, steel...",0.446154,58/130,0.015385,discriminating


In [8]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
    pipeline_schema=svc.pipeline_schema,
)

Selected best from 1 improving axes:
  max_token_candidates      best_delta=+58.3%  value_idx=0  acc=66.7%
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'fuzzy_matching': {'threshold': 70, 'scorer': 'WRatio'}, 'web_search': '<5 fields>', 'entity_profiling': '<5 fields>', 'token_matching': {'max_token_candidates': 20}, 'max_token_candidates': 10}

Composed winner: sp_hash=5217c2b40613

Round    Accuracy   Rolling Avg    Trend
  search     0.0%         0.0%  -


## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [9]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=scan_df,
    axis_profiles=axis_profiles,
    scan_variants=scan_variants,
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 0.0%
  Baseline prompt        : You are a comprehensive technical database API specialized in exhaustive entity ...
  ------------------------------------------------------------------
  Max rounds             : unlimited
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=2
  L3 (modify plan)       : enabled, patience=1
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : (default pipeline)
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE INPUT
     Prompt: You are a comprehensive technical database API specialized in exhaustive entit

In [10]:
#@title Run optimization (feedback cycle)
dev_reload()

campaign_rounds = await run_optimization_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc, pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)

2026-04-01 14:26:00 INFO     [api.services.campaign.optimization_loop] Using provided baseline (acc=0.000)
2026-04-01 14:26:00 INFO     [api.services.campaign.campaign_lifecycle] Cycle identity: cycle_e9315d106538
2026-04-01 14:26:00 INFO     [api.services.campaign.campaign_lifecycle] Resuming cycle cycle_e9315d106538 — 7 prior round(s) on disk


  Interrupt of cells can take up to 60 seconds!
  If a dialog pops up, click 'Cancel' and wait 20 seconds.

╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       0.0%                                               ║
║  Max rounds     999            Patience    2                       ║
║  Candidates     5                                                  ║
║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          openai/gpt-oss-120b                                ║
║  L2 (refine)    enabled            L3 (plan)   enabled             ║
║  Scan context   YES                                                ║
║  Critique       enabled                                            ║
╚═══════════════════════════════════════

2026-04-01 14:26:02 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-04-01 14:26:02 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-04-01 14:26:02 INFO     [api.services.campaign.campaign_lifecycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-04-01 14:26:02 INFO     [api.services.campaign.optimization_loop] Restored optimizer state from round 6 (critique=0 chars, task_context=6 keys, escalation_journal=0 entries, l2_round=0)
2026-04-01 14:26:02 INFO     [api.services.search.search_memory] SearchMemory refreshed: 6 new runs (total watermark: 206)
2026-04-01 14:26:02 INFO     [api.services.campaign.optimization_loop] Optimization round 0 (clean=0/999, acc=0.000, stall=2/2)
2026-04-01 14:26:02 INFO     [api.services.campaign.round_executio

  ✓ Initialized  cycle=cycle_e9315d  samples=15  obs=ON
    Resumed from round 7 (7 rounds cached)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ROUND 1/999                                               patience 0/2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

├─ GENERATE ─────────────────────────────────────────────────────────────┤
│  Current best    0.0%
│  Prompt          You are a comprehensive technical database API ...
│  Candidates      8   Creativity: 0.6   Scan: YES   Critique: NO
│  Model           openai/gpt-oss-120b
│  Scan focus: 1 improving axes [max_token_candidates]
├────────────────────────────────────────────────────────────────────────┤
  ✓ 5 candidates generated (loaded from disk)
    C1: Adopt a semantic‑focused thinking style to bett... [instruction]
    C2: Prepend an LCA‑specific query prefix to steer t... [instruction]
    C3: Raise profiling temperature to 0.7 to encourage... [instruction]
    C4:

2026-04-01 14:26:02 WARNING  [api.services.campaign.optimization_loop] Escalation 'degradation' at round 0 — target=l2, degraded_rate=100.0%


...

  ✓ L2 decision: 4 param changes, task_context updated, action=probe
    L2: The llm_ranking stage is completely unstable; adjusting temperatures and variant

  --- L2 PROMPT (sent to LLM) ---
  │ You are a prompt optimization expert.
  │ 
  │ The L1 inner optimization loop has stalled — candidates are no longer improving.
  │ 
  │ CURRENT PROMPT:
  │ ---
  │ You are a comprehensive technical database API specialized in exhaustive entity profiling.
  │ 
  │ Extract all possible information about a given query from research data and return it in a prescribed JSON format, including US/GB spelling variants, a single defining conceptual word, professional classification aliases, and detailed technical attributes.
  │ 
  │ Given a product or technical description query, parse unstructured research text to retrieve explicit specifications, infer implicit materials, processes, applications, and generate exhaustive arrays with spelling variants while identifying the core activity word.
  │ 
  

2026-04-01 14:26:10 INFO     [api.services.stores.campaign_store] Saved 4 candidates for round 1 → round_0001_candidates.json


  ✓ 4 candidates generated (from LLM)
    C1: Raise max_token_candidates to broaden candidate... [instruction]
    C2: Increase raw_content_limit to allow processing ... [instruction]
    C3: Add extra fields to profiling_schema to capture... [instruction]
    C4: Switch thinking_style to a detailed chain‑of‑th... [instruction]

│  Settings diff (5 params, 6 SPs):
│                        Start   Parent  C1      C2      C3      C4      
│  ─── entity_profiling ─
│      profiling_schema  -       -       -       -       [a]     -       
│     raw_content_limit  -       -       -       20000   -       -       
│  ─── token_matching ───
│  max_token_candidates  -       -       10      -       -       -       
│      entity_profiling  -       [b]     ·       ·       ·       ·       
│        thinking_style  -       -       -       -       -       [c]     
│  
│  Values:
│    [a] ['entity_name', 'core_concept', 'distinguishing_features', 'key_properties', 'technical_specifications', 'materia

2026-04-01 14:26:12 INFO     [api.services.campaign.layer_transitions] L2 refine_context: 3 param changes, task_context updated, action=probe, directive=290 chars
2026-04-01 14:26:12 INFO     [api.services.campaign.escalation] L2 requested probe — next round uses warned queries
2026-04-01 14:26:12 INFO     [api.services.campaign.escalation] L2 refine_context at round 1 (l2_round=2)
2026-04-01 14:26:12 INFO     [api.services.campaign.optimization_loop] PROBE round 2: 0 warned queries (from 0 tracked)
2026-04-01 14:26:12 INFO     [api.services.campaign.optimization_loop] Optimization round 2 (clean=1/999, acc=0.000, stall=2/2, PROBE)
2026-04-01 14:26:12 INFO     [api.services.campaign.round_execution] Loaded 5 persisted candidates for round 2
2026-04-01 14:26:12 INFO     [api.services.campaign.escalation] L3 patience exhausted (1 stalls) at round 2


  ✓ L2 decision: 4 param changes, task_context updated, action=probe
    L2: A broader candidate pool and clearer focus on decomposition and variant handling

  --- L2 PROMPT (sent to LLM) ---
  │ You are a prompt optimization expert.
  │ 
  │ The L1 inner optimization loop has stalled — candidates are no longer improving.
  │ 
  │ CURRENT PROMPT:
  │ ---
  │ You are a comprehensive technical database API specialized in exhaustive entity profiling.
  │ 
  │ Extract all possible information about a given query from research data and return it in a prescribed JSON format, including US/GB spelling variants, a single defining conceptual word, professional classification aliases, and detailed technical attributes.
  │ 
  │ Given a product or technical description query, parse unstructured research text to retrieve explicit specifications, infer implicit materials, processes, applications, and generate exhaustive arrays with spelling variants while identifying the core activity word.
  │ 
  

In [ ]:
#@title 5. Results — summary, save, sync
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=EXPERIMENT_ID,
)
sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)

CAMPAIGN SUMMARY (1 rounds)


,round,label,hit@1,total,accuracy,prompt_id
0,search,smart_search (scan_winner (sp_hash=5217c,0,0,0.0%,72ce39aed3c9


2026-04-01 14:26:13 INFO     [api.services.campaign.campaign_persistence] Winner saved: optimization/campaign_winner_72ce39aed3c9.json (acc=0.0%)


Need at least 2 rounds for flip tracking.
LINEAGE CHAIN
  [72ce39aed3c9] Round search: smart_search (scan_winner (sp_hash=5217c (0.0%)


2026-04-01 14:26:21 WARNING  [api.services.obs.langfuse_client] Failed to get Langfuse dataset
Traceback (most recent call last):
  File "C:\Users\dsacc\AppData\Roaming\Python\Python313\site-packages\langfuse\api\resources\dataset_items\client.py", line 246, in list
    _response_json = _response.json()
  File "C:\Users\dsacc\AppData\Roaming\Python\Python313\site-packages\httpx\_models.py", line 832, in json
    return jsonlib.loads(self.content, **kwargs)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Python313\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "c:\Python313\Lib\json\decoder.py", line 348, in decode
    raise JSONDecodeError("Extra data", s, end)
json.decoder.JSONDecodeError: Extra data: line 1 column 5 (char 4)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\dsacc\Desktop\PromptPotter\prompt-potter-optimizer\